## Hotel Recommendation System

#### Objective

Build a hotel recommendation system using historical booking data.

The recommendation engine will:

- Analyze user booking behavior
- Identify similar users
- Recommend hotels based on booking history

The generated recommendations will later be integrated into a Streamlit application.

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

import joblib
import os

#### Load Dataset

Load the processed hotel booking dataset.

In [2]:
hotel_user = pd.read_csv(
    "../data/processed/hotel_user.csv"
)

hotel_user.head()

,travelCode,userCode,hotel_name,place,days,price,total,date,booking_month,code,company,name,gender,age
0,0,0,Hotel A,Florianopolis (SC),4,313.02,1252.08,2019-09-26,9,0,4You,Roy Braun,male,21
1,2,0,Hotel K,Salvador (BH),2,263.41,526.82,2019-10-10,10,0,4You,Roy Braun,male,21
2,7,0,Hotel K,Salvador (BH),3,263.41,790.23,2019-11-14,11,0,4You,Roy Braun,male,21
3,11,0,Hotel K,Salvador (BH),4,263.41,1053.64,2019-12-12,12,0,4You,Roy Braun,male,21
4,13,0,Hotel A,Florianopolis (SC),1,313.02,313.02,2019-12-26,12,0,4You,Roy Braun,male,21


#### Dataset Overview

In [3]:
print("Dataset Shape:", hotel_user.shape)

hotel_user.info()

Dataset Shape: (40552, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40552 entries, 0 to 40551
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   travelCode     40552 non-null  int64  
 1   userCode       40552 non-null  int64  
 2   hotel_name     40552 non-null  object 
 3   place          40552 non-null  object 
 4   days           40552 non-null  int64  
 5   price          40552 non-null  float64
 6   total          40552 non-null  float64
 7   date           40552 non-null  object 
 8   booking_month  40552 non-null  int64  
 9   code           40552 non-null  int64  
 10  company        40552 non-null  object 
 11  name           40552 non-null  object 
 12  gender         40552 non-null  object 
 13  age            40552 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 4.3+ MB


## Create Hotel Recommendation Dataset

Aggregate hotel booking statistics by destination and hotel.

This information will be used to recommend hotels for specific destinations.

In [4]:
hotel_recommendation_df = (
    hotel_user
    .groupby(
        ["place", "hotel_name"]
    )
    .agg(
        booking_count=("travelCode", "count"),
        avg_price=("price", "mean"),
        avg_stay_days=("days", "mean"),
        avg_total_spend=("total", "mean")
    )
    .reset_index()
)

hotel_recommendation_df

,place,hotel_name,booking_count,avg_price,avg_stay_days,avg_total_spend
0,Aracaju (SE),Hotel Z,4205,208.04,2.491558,518.343658
1,Brasilia (DF),Hotel BP,4437,247.62,2.486365,615.673617
2,Campo Grande (MS),Hotel BW,4333,60.39,2.495500,150.703224
3,Florianopolis (SC),Hotel A,3330,313.02,2.483183,777.286000
4,Natal (RN),Hotel BD,4829,242.88,2.499068,606.973667
5,Recife (PE),Hotel AU,4467,312.83,2.520931,788.622930
6,Rio de Janeiro (RJ),Hotel CB,5029,165.99,2.485981,412.648037
7,Salvador (BH),Hotel K,5094,263.41,2.516490,662.868628
8,Sao Paulo (SP),Hotel AF,4828,139.10,2.511599,349.363422


#### Recommendation Function

In [5]:
def recommend_hotel(
    destination
):

    recommendations = (
        hotel_recommendation_df[
            hotel_recommendation_df["place"] == destination
        ]
        .sort_values(
            by="booking_count",
            ascending=False
        )
    )

    return recommendations

#### Example Recommendation

In [6]:
recommend_hotel(
    "Salvador (BH)"
)

,place,hotel_name,booking_count,avg_price,avg_stay_days,avg_total_spend
7,Salvador (BH),Hotel K,5094,263.41,2.51649,662.868628


In [7]:
import joblib
import os

os.makedirs(
    "../models",
    exist_ok=True
)

joblib.dump(
    hotel_recommendation_df,
    "../models/hotel_recommendation.pkl"
)

print(
    "Hotel Recommendation Dataset Saved"
)

Hotel Recommendation Dataset Saved


Why didn't you use collaborative filtering?

The dataset contains only 9 hotels, with each hotel uniquely associated with a destination. Since users are not choosing among multiple hotels within the same destination, collaborative filtering does not provide meaningful recommendations. Therefore, a destination-based recommendation system was implemented using booking frequency and hotel performance metrics.